In [ ]:
import sys
import os

os.environ["CUDA_VISIBLE_DEVICES"]="0"

# 必须包含 ops 目录，因为 selective_scan_interface 通常在那
paths = [
    "/home/xiaolei/projects/baseline/ReID/CLIMB-ReID/mamba",
    "/home/xiaolei/projects/baseline/ReID/CLIMB-ReID/mamba/mamba_ssm/ops"
]

for p in paths:
    if p not in sys.path:
        sys.path.insert(0, p)

# 打印出来确认一下
print("\n".join(sys.path[:3]))

In [ ]:
from utils.logger import setup_logger
import random
import torch
import torch.nn as nn
import numpy as np
import os
import argparse
from config import cfg
from solver.lr_scheduler import WarmupMultiStepLR
from climb.dataloader import make_CLIMB_dataloader
from climb.processor_climb import do_inference
from climb.optimizer import make_CLIMB_optimizer
from climb.model import make_model

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
if __name__ == '__main__':

    # 在执行任何 argparse 解析之前加上这一段
    if 'ipykernel_launcher' in sys.argv[0]:
        # 只保留脚本名，清空所有干扰参数
        sys.argv = [sys.argv[0]]

    parser = argparse.ArgumentParser(description="ReID Baseline Training")
    parser.add_argument(
        "--config_file", default="./config/climb-vit-market.yml", help="path to config file", type=str
    )

    parser.add_argument("opts", help="Modify config options using the command-line", default=None,
                        nargs=argparse.REMAINDER)
    parser.add_argument("--local_rank", default=0, type=int)
    # args = parser.parse_args()
    args, args_remain = parser.parse_known_args()

    if args.config_file != "":
        cfg.merge_from_file(args.config_file)
    cfg.merge_from_list(args.opts)
    cfg.freeze()

    set_seed(cfg.SOLVER.SEED)
    
    output_dir = cfg.OUTPUT_DIR
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    logger = setup_logger("CLIMB", output_dir, if_train=True)
    logger.info("Saving model in the path :{}".format(cfg.OUTPUT_DIR))
    logger.info(args)
    
    if args.config_file != "":
        logger.info("Loaded configuration file {}".format(args.config_file))
        with open(args.config_file, 'r') as cf:
            config_str = "\n" + cf.read()
            logger.info(config_str)
    logger.info("Running with config:\n{}".format(cfg))
    
    train_loader, val_loader, cluster_loader, num_query, num_classes, camera_num, view_num = make_CLIMB_dataloader(cfg)
    model = make_model(cfg, num_classes, camera_num=camera_num, view_num=view_num)
    model.load_param(cfg.TEST.PRETRAINED_PATH)
    optimizer = make_CLIMB_optimizer(cfg, model)
    scheduler = WarmupMultiStepLR(optimizer, cfg.SOLVER.STEPS, cfg.SOLVER.GAMMA, cfg.SOLVER.WARMUP_FACTOR,
                                  cfg.SOLVER.WARMUP_ITERS, cfg.SOLVER.WARMUP_METHOD)
    
    do_inference(cfg, model, val_loader, num_query)

In [ ]:
if __name__ == '__main__':

    # 在执行任何 argparse 解析之前加上这一段
    if 'ipykernel_launcher' in sys.argv[0]:
        # 只保留脚本名，清空所有干扰参数
        sys.argv = [sys.argv[0]]

    parser = argparse.ArgumentParser(description="ReID Baseline Training")
    parser.add_argument(
        "--config_file", default="./config/climb-vit-market.yml", help="path to config file", type=str
    )

    parser.add_argument("opts", help="Modify config options using the command-line", default=None,
                        nargs=argparse.REMAINDER)
    parser.add_argument("--local_rank", default=0, type=int)
    # args = parser.parse_args()
    args, args_remain = parser.parse_known_args()

    if args.config_file != "":
        cfg.merge_from_file(args.config_file)
    cfg.merge_from_list(args.opts)
    cfg.freeze()

    set_seed(cfg.SOLVER.SEED)
    
    output_dir = cfg.OUTPUT_DIR
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    logger = setup_logger("CLIMB", output_dir, if_train=True)
    logger.info("Saving model in the path :{}".format(cfg.OUTPUT_DIR))
    logger.info(args)
    
    if args.config_file != "":
        logger.info("Loaded configuration file {}".format(args.config_file))
        with open(args.config_file, 'r') as cf:
            config_str = "\n" + cf.read()
            logger.info(config_str)
    logger.info("Running with config:\n{}".format(cfg))
    
    train_loader, val_loader, cluster_loader, num_query, num_classes, camera_num, view_num = make_CLIMB_dataloader(cfg)
    model = make_model(cfg, num_classes, camera_num=camera_num, view_num=view_num)
    model.load_param(cfg.TEST.PRETRAINED_PATH)

    device = "cuda"
    batch_size = 1
    channels = 3
    height = 256
    width = 128
    camera_num = 6
    imgs = torch.randn(batch_size, channels, height, width)
    target_cam = torch.randint(0, camera_num, (batch_size,))
    print(f'imgs: {imgs.shape}, target_cam: {target_cam.shape}')
    model.to(device)
    model = nn.DataParallel(model).cuda()
    model.eval()
    img = imgs.to(device)
    camids = target_cam.to(device)
    feat, feat1, feat2 = model(img, cam_label=camids, view_label=None)
    print(f'feat: {feat.shape}, feat1: {feat1.shape}, feat2: {feat2.shape}')

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 模拟CLIMB模型的前向过程
def simulate_climb_forward():
    """
    模拟CLIMB模型的前向传播过程，分析每个特征的维度
    假设使用ViT-B/16作为主干网络
    """
    
    # ==================== 初始化参数 ====================
    print("=" * 60)
    print("CLIMB模型前向过程模拟")
    print("=" * 60)
    
    # 假设输入图像大小
    input_height = 256
    input_width = 128
    batch_size = 64  # 假设batch_size=64
    
    # CLIP ViT-B/16 参数
    patch_size = 16
    stride_size = 16
    width = 768  # 特征维度
    output_dim = 512  # 投影维度
    
    # 计算网格大小 (h_resolution, w_resolution)
    h_resolution = (input_height - 16) // stride_size + 1
    w_resolution = (input_width - 16) // stride_size + 1
    num_patches = h_resolution * w_resolution
    print(f"输入图像尺寸: ({input_height}, {input_width})")
    print(f"网格分辨率: ({h_resolution}, {w_resolution})")
    print(f"patch数量: {num_patches}")
    print(f"batch_size: {batch_size}")
    print()
    
    # ==================== 模拟输入 ====================
    # 模拟输入图像 [B, 3, H, W]
    x = torch.randn(batch_size, 3, input_height, input_width)
    print(f"1. 输入图像维度: {x.shape}")
    print(f"   形状: [batch_size={batch_size}, channels=3, height={input_height}, width={input_width}]")
    print()
    
    # 模拟相机和视角标签（可选）
    cam_label = torch.randint(0, 8, (batch_size,))  # 假设8个相机
    view_label = torch.randint(0, 5, (batch_size,)) # 假设5个视角
    
    # ==================== CLIP图像编码器模拟 ====================
    print("2. 通过CLIP图像编码器 (ViT-B/16)")
    print("   " + "-" * 40)
    
    # 模拟conv1层：提取patch embeddings
    # 输入: [B, 3, H, W] -> 输出: [B, 768, h_resolution, w_resolution]
    conv_out = torch.randn(batch_size, width, h_resolution, w_resolution)
    print(f"   a) Conv1后: {conv_out.shape}")
    print(f"      形状: [B={batch_size}, D={width}, H={h_resolution}, W={w_resolution}]")
    
    # reshape: [B, D, h_resolution, w_resolution] -> [B, D, num_patches]
    conv_out_reshaped = conv_out.view(batch_size, width, num_patches)
    print(f"   b) Reshape后: {conv_out_reshaped.shape}")
    print(f"      形状: [B={batch_size}, D={width}, N={num_patches}]")
    
    # permute: [B, D, N] -> [B, N, D]
    patch_embeddings = conv_out_reshaped.permute(0, 2, 1)
    print(f"   c) Permute后 (patch embeddings): {patch_embeddings.shape}")
    print(f"      形状: [B={batch_size}, N={num_patches}, D={width}]")
    
    # 添加[CLS] token: [B, 1, D]
    cls_token = torch.randn(batch_size, 1, width)
    image_features = torch.cat([cls_token, patch_embeddings], dim=1)
    print(f"   d) 添加[CLS] token后: {image_features.shape}")
    print(f"      形状: [B={batch_size}, N+1={num_patches+1}, D={width}]")
    print(f"      (包含1个[CLS] token和{num_patches}个patch token)")
    
    # 模拟投影输出 [B, N+1, output_dim]
    image_features_proj = torch.randn(batch_size, num_patches + 1, output_dim)
    print(f"   e) 投影特征: {image_features_proj.shape}")
    print(f"      形状: [B={batch_size}, N+1={num_patches+1}, D_proj={output_dim}]")
    print()
    
    # ==================== 提取[CLS] token ====================
    print("3. 提取[CLS] token特征")
    print("   " + "-" * 40)
    
    img_feature = image_features[:, 0, :]  # [B, D]
    img_feature_proj = image_features_proj[:, 0, :]  # [B, D_proj]
    
    print(f"   a) ViT特征 [CLS]: {img_feature.shape}")
    print(f"      形状: [B={batch_size}, D={width}]")
    print(f"   b) 投影特征 [CLS]: {img_feature_proj.shape}")
    print(f"      形状: [B={batch_size}, D_proj={output_dim}]")
    print()
    
    # ==================== Bottleneck处理 ====================
    print("4. Bottleneck标准化处理")
    print("   " + "-" * 40)
    
    # 模拟BatchNorm1d层 (in_planes=768, in_planes_proj=512)
    feat = torch.randn(batch_size, width)  # 模拟bottleneck输出
    feat_proj = torch.randn(batch_size, output_dim)  # 模拟bottleneck_proj输出
    
    print(f"   a) ViT特征bottleneck后: {feat.shape}")
    print(f"   b) 投影特征bottleneck后: {feat_proj.shape}")
    
    # 特征拼接
    out_feat = torch.cat([feat, feat_proj], dim=1)
    print(f"   c) 拼接特征: {out_feat.shape}")
    print(f"      形状: [B={batch_size}, {width + output_dim}]")
    print(f"      (768维ViT特征 + 512维投影特征)")
    print()
    
    # ==================== Mamba分支处理 ====================
    print("5. Mamba分支处理 (空间建模)")
    print("   " + "-" * 40)
    
    # 分离特征用于Mamba
    feats_for_mamba = image_features.detach()
    print(f"   a) 分离特征用于Mamba: {feats_for_mamba.shape}")
    
    # 分离patch tokens (排除[CLS] token)
    feats_for_mamba_sp = feats_for_mamba[:, 1:, :]  # [B, N, D]
    feats_for_mamba_cls = feats_for_mamba[:, 0, :]   # [B, D]
    
    print(f"   b) Patch tokens: {feats_for_mamba_sp.shape}")
    print(f"      形状: [B={batch_size}, N={num_patches}, D={width}]")
    print(f"   c) [CLS] token: {feats_for_mamba_cls.shape}")
    print(f"      形状: [B={batch_size}, D={width}]")
    print()
    
    # ==================== 软重排序 ====================
    print("6. 基于[CLS] token的软重排序")
    print("   " + "-" * 40)
    
    # 模拟软重排序过程
    # 计算patch tokens与[CLS] token的相似度
    ref_norm = torch.randn(batch_size, width)  # 归一化的[CLS] token
    raw_norm = torch.randn(batch_size, num_patches, width)  # 归一化的patch tokens
    
    # 相似度计算: [B, N]
    scores = torch.einsum('bc,bnc->bn', ref_norm, raw_norm)
    print(f"   a) Patch重要度分数: {scores.shape}")
    print(f"      形状: [B={batch_size}, N={num_patches}]")
    
    # 软重排序后的patch tokens
    re_order_mamba_sp = torch.randn(batch_size, num_patches, width)
    print(f"   b) 重排序后patch tokens: {re_order_mamba_sp.shape}")
    print(f"      形状: [B={batch_size}, N={num_patches}, D={width}]")
    print()
    
    # ==================== Mamba处理 ====================
    print("7. BiMamba处理")
    print("   " + "-" * 40)
    
    # 通过BiMamba
    mamba_sp_out = torch.randn(batch_size, num_patches, width)
    print(f"   a) BiMamba输出: {mamba_sp_out.shape}")
    
    # 重新添加[CLS] token
    mamba_sp_out = torch.cat([feats_for_mamba_cls.unsqueeze(1), mamba_sp_out], dim=1)
    print(f"   b) 添加[CLS] token后: {mamba_sp_out.shape}")
    print(f"      形状: [B={batch_size}, N+1={num_patches+1}, D={width}]")
    print()
    
    # ==================== 空间注意力池化 ====================
    print("8. 空间注意力池化")
    print("   " + "-" * 40)
    
    # LayerNorm
    mamba_sp_out2 = torch.randn(batch_size, num_patches + 1, width)
    print(f"   a) LayerNorm后: {mamba_sp_out2.shape}")
    
    # 空间注意力
    # sp_attention: Linear(768->192) -> Tanh -> Linear(192->1)
    A = torch.randn(batch_size, num_patches + 1, 1)  # [B, N+1, 1]
    print(f"   b) 注意力权重: {A.shape}")
    print(f"      形状: [B={batch_size}, N+1={num_patches+1}, 1]")
    
    # 转置和softmax
    A = A.permute(0, 2, 1)  # [B, 1, N+1]
    A = torch.softmax(A, dim=-1)
    print(f"   c) Softmax后注意力: {A.shape}")
    
    # 加权平均
    weighted_feat = torch.bmm(A, mamba_sp_out2)  # [B, 1, D]
    print(f"   d) 加权平均: {weighted_feat.shape}")
    print(f"      形状: [B={batch_size}, 1, D={width}]")
    
    # squeeze
    mamba_sp_out2 = weighted_feat.squeeze(1)  # [B, D]
    print(f"   e) 最终Mamba特征: {mamba_sp_out2.shape}")
    print(f"      形状: [B={batch_size}, D={width}]")
    print()
    
    # ==================== 输出汇总 ====================
    print("9. 最终输出汇总")
    print("   " + "-" * 40)
    
    # Mamba特征的bottleneck
    feat_sp = torch.randn(batch_size, width)
    
    print(f"   a) 主分支特征 (CLIP): {out_feat.shape}")
    print(f"      维度: {width + output_dim} = {width}(ViT) + {output_dim}(投影)")
    
    print(f"   b) Mamba分支特征: {feat_sp.shape}")
    print(f"      维度: {width}")
    
    # 训练时输出
    if True:  # 假设在训练模式下
        # 分类器输出
        logit = torch.randn(batch_size, num_classes := 1000)  # 假设1000个类别
        logitsp = torch.randn(batch_size, num_classes)
        
        print(f"\n   训练模式输出:")
        print(f"     1) out_feat: {out_feat.shape}")
        print(f"     2) logit: {logit.shape} (分类logits)")
        print(f"     3) feat_sp: {feat_sp.shape}")
        print(f"     4) logitsp: {logitsp.shape} (Mamba分类logits)")
    
    # 测试时输出
    else:
        feat_concat = torch.cat([out_feat, feat_sp], dim=1)
        print(f"\n   测试模式输出:")
        print(f"     1) feat_concat: {feat_concat.shape} (拼接特征)")
        print(f"       维度: {out_feat.shape[1] + feat_sp.shape[1]} = {out_feat.shape[1]} + {feat_sp.shape[1]}")
        print(f"     2) out_feat: {out_feat.shape} (CLIP特征)")
        print(f"     3) feat_sp: {feat_sp.shape} (Mamba特征)")
    
    print()
    print("=" * 60)
    print("前向过程模拟完成")
    print("=" * 60)
    
    # 返回模拟的特征维度字典
    return {
        "input": x.shape,
        "image_features": image_features.shape,
        "image_features_proj": image_features_proj.shape,
        "img_feature": img_feature.shape,
        "img_feature_proj": img_feature_proj.shape,
        "out_feat": out_feat.shape,
        "feats_for_mamba": feats_for_mamba.shape,
        "mamba_sp_out": mamba_sp_out.shape,
        "feat_sp": feat_sp.shape,
        "final_concat": torch.cat([out_feat, feat_sp], dim=1).shape if not True else None
    }

# 运行模拟
if __name__ == "__main__":
    # 模拟前向过程
    dim_dict = simulate_climb_forward()
    
    # 输出维度总结表
    print("\n" + "="*80)
    print("维度总结表")
    print("="*80)
    print(f"{'阶段':<25} {'形状':<30} {'说明':<30}")
    print("-"*80)
    
    stages = [
        ("输入图像", "input"),
        ("ViT特征 (全token)", "image_features"),
        ("投影特征 (全token)", "image_features_proj"),
        ("ViT [CLS] token", "img_feature"),
        ("投影 [CLS] token", "img_feature_proj"),
        ("主分支特征 (拼接后)", "out_feat"),
        ("Mamba输入特征", "feats_for_mamba"),
        ("Mamba输出特征", "mamba_sp_out"),
        ("Mamba分支特征", "feat_sp"),
    ]
    
    for name, key in stages:
        if key in dim_dict and dim_dict[key] is not None:
            shape_str = str(dim_dict[key])
            description = {
                "input": "原始输入图像",
                "image_features": "ViT提取的所有token特征",
                "image_features_proj": "投影后的所有token特征", 
                "img_feature": "ViT的[CLS] token特征",
                "img_feature_proj": "投影后的[CLS] token特征",
                "out_feat": "CLIP特征 + 投影特征",
                "feats_for_mamba": "分离给Mamba的特征",
                "mamba_sp_out": "Mamba处理后的特征",
                "feat_sp": "Mamba分支的池化特征"
            }.get(key, "")
            print(f"{name:<25} {shape_str:<30} {description:<30}")